# Can a delivery-app star rating tell you which Hanoi restaurant is good?

Data: one snapshot of ShopeeFood's Hanoi listing (2026-07-25), 402 restaurants,
plus 693 Foody reviews covering 115 of them.

Four limits, stated before any number:

1. This is the promoted, deliverable-to-one-address slice of the listing, not
   all of Hanoi. Every card carried a voucher badge.
2. The reviews are 2018-2021. Foody's review flow effectively stopped when
   ShopeeFood took over ordering. They describe what diners complained about
   then, not service quality today.
3. Review coverage reaches 115 of 402 restaurants, and those 115 skew popular
   (median review-count bucket 100 against 10 for the rest).
4. `review_count` is a display bucket, not a count. It is used as a popularity
   tier and nowhere else.

In [1]:
import json, pandas as pd, numpy as np

SNAPSHOT = "snapshot_20260725T061403.json"   # upload to Colab; never committed
raw = json.load(open(SNAPSHOT, encoding="utf-8"))

# Capture batches overlap, so the collector emits duplicate rows on purpose
# (parse-only boundary). Dedupe is the notebook's job.
rest = pd.DataFrame(raw["restaurants"]).drop_duplicates("restaurant_id")

# Foody ignores ?page=N on /binh-luan and re-served page 1, so every review
# appears exactly twice. Not deduping would double every count.
rev = pd.DataFrame(raw["reviews"]).drop_duplicates("review_id")

assert len(rest) == 402, len(rest)
assert len(rev) == 693, len(rev)
print(f"{len(rest)} restaurants, {len(rev)} reviews")

402 restaurants, 693 reviews


In [2]:
# rating 0 means unrated, not badly rated. 52 restaurants.
rest["rating"] = rest["rating"].replace(0, np.nan)

# District is the second-to-last comma field of the address. No id->name
# lookup table needed, and it reads better than district_id.
rest["district"] = (rest["address"].str.split(",")
                    .str[-2].str.strip())

# `categories` is a venue type, not a cuisine. `cuisine_raw` covers only half
# the rows and 85% of those say "Mon Viet", so it has no separating power --
# venue type is the honest axis, and the page names it that way.
rest["category"] = rest["categories"].str[0]

# Analysable set: a price band AND a real rating.
A = rest.dropna(subset=["rating"]).query("price_max > 0").copy()
assert len(A) == 337, len(A)

print(rest["district"].value_counts())
print(rest["category"].value_counts().head(6))

district
Đống Đa         100
Hai Bà Trưng     91
Ba Đình          67
Hoàn Kiếm        44
Cầu Giấy         42
Thanh Xuân       33
Hoàng Mai        24
Long Biên         1
Name: count, dtype: int64
category
Quán ăn          236
Café/Dessert      67
Shop Online       49
Ăn vặt/vỉa hè     30
Nhà hàng          14
Tiệm bánh          2
Name: count, dtype: int64


In [3]:
print(A["price_max"].describe(percentiles=[.1,.25,.5,.75,.9,.95,.99]))
print(A.nlargest(8, "price_max")[["name","category","price_max"]])

count        337.00000
mean       91344.21365
std       111572.42031
min        10000.00000
10%        30000.00000
25%        40000.00000
50%        53000.00000
75%       100000.00000
90%       199000.00000
95%       300000.00000
99%       500000.00000
max      1000000.00000
Name: price_max, dtype: float64
                                                  name   category  price_max
146                  Dim Sum Corner - Ẩm Thực Hongkong   Nhà hàng  1000000.0
175                Cousins - Ẩm Thực Châu Âu - Đào Tấn   Nhà hàng  1000000.0
448  Hiệu Lực - Canh Cá Rô Đồng Hưng Yên - Hai Bà T...    Quán ăn   600000.0
27               Bảo Minh - Đặc Sản Bánh Cốm Hàng Than  Tiệm bánh   500000.0
31                                        Lẩu Bee Phạm    Quán ăn   500000.0
160                      Dũng Huyền - Ngan Ngon Phố Cổ    Quán ăn   500000.0
119             Khao Thai - Tiệm Ăn Thái Lan - Cửa Bắc    Quán ăn   428000.0
173        Tonchan Ramen - Ẩm Thực Nhật - Bùi Thị Xuân   Nhà hàng   400000.0

In [4]:
A["price_band"] = pd.qcut(A["price_max"], 4,
                          labels=["cheapest", "lower-mid", "upper-mid", "priciest"])

In [5]:
rev["rating"] = pd.to_numeric(rev["rating"], errors="coerce")
rev = rev.dropna(subset=["rating"])          # 2 reviews carry no score
rev["year"] = pd.to_datetime(rev["created_at"], format="mixed",
                             utc=True).dt.year
assert len(rev) == 691, len(rev)
print(rev["year"].value_counts().sort_index())

year
2015      1
2016     30
2017     80
2018    119
2019    188
2020    177
2021     69
2022     15
2023      7
2024      3
2025      2
Name: count, dtype: int64
